Distribuição das classes no Fundão e em Badajoz

In [ ]:
from pathlib import Path
import sys
import geopandas as gpd
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import ANALYSIS
from classification_utils import class_area_table

In [ ]:
pilots = ["fundao", "badajoz"]
all_classes = []
all_summary = []

for pilot in pilots:
    pilot_dir = ANALYSIS / "02_classes" / pilot
    inventory = pd.read_csv(
        pilot_dir / f"class_inventory_{pilot}.csv"
    )

    municipality_file = Path(inventory.iloc[0]["municipality"])
    municipality = gpd.read_file(
        municipality_file,
        layer="municipality"
    )

    import rasterio as rio

    with rio.open(inventory.iloc[0]["local_class_raster"]) as src:
        municipality = municipality.to_crs(src.crs)

    municipality_area_km2 = (
        municipality.geometry.area.sum() / 1_000_000
    )

    for _, record in inventory.iterrows():
        table = class_area_table(
            record["local_class_raster"],
            pilot=pilot,
            area=record["area"],
            map_id=record["map_id"],
            scenario=record["scenario"],
            product=record["product"],
            method=record["method"]
        )

        modelled_area = table["area_km2"].sum()
        table["municipality_area_km2"] = municipality_area_km2
        table["modelled_area_km2"] = modelled_area
        table["coverage_pct_municipality"] = (
            100.0 * modelled_area / municipality_area_km2
        )

        all_classes.append(table)

        high_very_high = table.loc[
            table["class_code"] >= 4,
            "area_km2"
        ].sum()

        all_summary.append({
            "pilot": pilot,
            "area": record["area"],
            "map_id": record["map_id"],
            "scenario": record["scenario"],
            "product": record["product"],
            "method": record["method"],
            "municipality_area_km2": municipality_area_km2,
            "modelled_area_km2": modelled_area,
            "coverage_pct_municipality": (
                100.0 * modelled_area / municipality_area_km2
            ),
            "high_very_high_km2": high_very_high,
            "high_very_high_pct_modelled": (
                100.0 * high_very_high / modelled_area
                if modelled_area
                else None
            )
        })

In [ ]:
class_distribution = pd.concat(all_classes, ignore_index=True)
priority_summary = pd.DataFrame(all_summary)

out_dir = ANALYSIS / "03_pilot_class_distribution"
out_dir.mkdir(parents=True, exist_ok=True)
output = out_dir / "pilot_class_distribution.xlsx"

with pd.ExcelWriter(output) as writer:
    class_distribution.to_excel(
        writer,
        sheet_name="Classes",
        index=False
    )
    priority_summary.to_excel(
        writer,
        sheet_name="High_very_high",
        index=False
    )

print("Resultados guardados em:", output)
priority_summary